In [ ]:
# Установка Gymnasium, SB3 и зависимостей для рендеринга
!pip install gymnasium[lunar-lander] stable-baselines3[extra] shimmy pyvirtualdisplay
!sudo apt-get install -y xvfb python-opengl ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 19.1 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package python-opengl


In [ ]:
!pip install gymnasium[box2d]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 103.1 MB/s eta 0:00:00


In [ ]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

# Создаем среду
env = gym.make("LunarLander-v3", render_mode="rgb_array")
env = Monitor(env)

# Инициализация модели PPO (MlpPolicy — обычный перцептрон для работы с векторами)
model_ppo = PPO("MlpPolicy", env, verbose=1, n_steps=8192, batch_size=128, learning_rate = 2e-4, tensorboard_log="./ppo_lander_logs/")

# Обучение
model_ppo.learn(total_timesteps=300000)
model_ppo.save("ppo_lunar")

In [ ]:
model_dqn = DQN(
    "MlpPolicy",
    env,
    learning_rate=5e-4,
    buffer_size=100000,
    learning_starts=2000,
    exploration_fraction=0.5, # Долго и упорно исследуем мир
    target_update_interval=1000,
    verbose=1
)

model_dqn.learn(total_timesteps=300000)
model_dqn.save("dqn_lunar")

NameError: name 'DQN' is not defined

In [ ]:
import gymnasium as gym
from sb3_contrib import QRDQN

# 1. Используем ДИСКРЕТНУЮ среду (без слова Continuous)
env = gym.make("LunarLander-v3", render_mode="rgb_array")

# 2. Инициализация QRDQN
# n_steps=5 — это отличный выбор (multi-step learning из Rainbow)
model_qrdqn = QRDQN(
    "MlpPolicy",
    env,
    n_steps=5,
    verbose=1,
    learning_rate=1e-3, # QR-DQN обычно "переваривает" LR повыше
    policy_kwargs=dict(n_quantiles=50), # Количество квантилей для аппроксимации распределения
    tensorboard_log="./qrdqn_lander_logs/"
)

# 3. Обучение
# С параметром n_steps=5 он должен обучаться быстрее обычного DQN
model_qrdqn.learn(total_timesteps=300000, log_interval=10)

# 4. Сохранение
model_qrdqn.save("qrdqn_lunar")

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise

# 1. Создаем НЕПРЕРЫВНУЮ версию среды
env = gym.make("LunarLanderContinuous-v3", render_mode="rgb_array")

# 2. Настраиваем шум (exploration)
# Количество шума должно соответствовать количеству выходных действий (их 2 в этой среде)
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# 3. Инициализируем модель
# DDPG обычно требует чуть больше памяти (buffer_size) и специфический LR
model_ddpg = DDPG(
    "MlpPolicy",
    env,
    action_noise=action_noise,
    verbose=100,
    learning_rate=2e-4,
    buffer_size=200000,
    batch_size=128,
    tau=0.005, # Мягкое обновление целевой сети (Soft update)
    tensorboard_log="./ddpg_lander_logs/"
)

# 4. Обучаем (для непрерывной посадки нужно минимум 300к шагов)
model_ddpg.learn(total_timesteps=500000, log_interval=5)
model_ddpg.save("ddpg_lunar")

In [ ]:
import gymnasium as gym
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor

# 1. Создаем среду
env = gym.make("LunarLanderContinuous-v3", render_mode="rgb_array")
env = Monitor(env)

# 2. Инициализация SAC
# SAC сам настраивает коэффициент энтропии (ent_coef='auto'),
# что избавляет нас от лишней головной боли.
model_sac = SAC(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    buffer_size=1000000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    learning_starts=1000,
    tensorboard_log="./sac_lander_logs/"
)

# 3. Обучение
# Для Lunar Lander 300к-500к шагов обычно достаточно для идеальной посадки
model_sac.learn(total_timesteps=300000, log_interval=10)

# 4. Сохранение
model_sac.save("sac_lunar_lander")

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Создаем группу параллельных сред
NUM_ENVS = 8  # Размер группы (G в GRPO)
envs = gym.vector.Make("LunarLander-v3", num_envs=NUM_ENVS)

class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.net(x)

# Инициализация
state_dim = envs.single_observation_space.shape[0]
action_dim = envs.single_action_space.n
policy = PolicyNetwork(state_dim, action_dim)
optimizer = optim.AdamW(policy.parameters(), lr=1e-4)

In [ ]:
def train_grpo_step(states, actions, rewards):
    states = torch.FloatTensor(states)
    actions = torch.LongTensor(actions)
    rewards = torch.FloatTensor(rewards)

    # --- СЕРДЦЕ GRPO: Групповой расчет Advantage ---
    # Вычисляем среднее и стандартное отклонение внутри группы (по оси сред)
    mean_reward = rewards.mean()
    std_reward = rewards.std() + 1e-8
    advantages = (rewards - mean_reward) / std_reward
    # ----------------------------------------------

    # Получаем вероятности действий от текущей политики
    probs = policy(states)
    action_probs = probs.gather(1, actions.unsqueeze(1)).squeeze()

    # Loss функция (как в PPO, но без Критика)
    loss = -(torch.log(action_probs) * advantages).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
total_steps = 100000
obs, _ = envs.reset()

for step in range(total_steps // NUM_ENVS):
    # 1. Сбор опыта группой
    state_tensor = torch.FloatTensor(obs)
    with torch.no_grad():
        action_probs = policy(state_tensor)
        # Выбираем действия на основе вероятностей
        actions = torch.multinomial(action_probs, 1).squeeze().numpy()

    next_obs, rewards, terminations, truncations, infos = envs.step(actions)

    # 2. Шаг обучения по логике GRPO
    loss_val = train_grpo_step(obs, actions, rewards)

    obs = next_obs

    if step % 100 == 0:
        print(f"Step: {step*NUM_ENVS}, Loss: {loss_val:.4f}, Mean Reward: {rewards.mean():.2f}")

In [ ]:
!pip install unsloth "trl<0.15.0" peft accelerate bitsandbytes

In [ ]:
from datasets import load_dataset

# Загружаем бинаризованный UltraFeedback
# Мы берем только 2000 примеров, чтобы обучение в Колабе не шло вечность
dataset = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs[:2000]")

def format_dpo_dataset(examples):
    # UltraFeedback имеет сложную структуру, нам нужно достать промпт,
    # выбранный ответ (chosen) и отвергнутый (rejected)
    new_examples = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
    }

    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        # Вытаскиваем только текст ответа из структуры сообщений
        # Обычно это последнее сообщение в списке
        new_examples["prompt"].append(prompt)
        new_examples["chosen"].append(chosen[-1]["content"])
        new_examples["rejected"].append(rejected[-1]["content"])

    return new_examples

# Преобразуем формат
dataset = dataset.map(format_dpo_dataset, batched=True, remove_columns=dataset.column_names)

# Посмотрим, что получилось
print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'prompt': 'how can i develop a habit of drawing daily', 'chosen': "Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:\n\n1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.\n2. Set a specific duration: Determine the amount of time you want to spend on drawing each day. It can be as little as 10 minutes or as long as an hour. Be consistent with the duration to help build the habit.\n3. Start small and simple: Don't try to create a masterpiece every day, start with simple and easy-to-do sketches. Focus on improving your skills gradually.\n4. Use a variety of tools and mediums: Experiment with different tools like pencils, pens, markers, and different mediums like paper, canvas, or digital a

In [ ]:
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
import torch

# Загружаем модель (возьмем Qwen 1.5B, она очень умная для своего размера)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
)

# Конфиг для "боевого" теста
dpo_config = DPOConfig(
    per_device_train_batch_size = 4, # Увеличили батч, Unsloth вывезет
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    max_steps = 150,
    learning_rate = 5e-6, # Для DPO на реальных данных лучше брать LR поменьше
    fp16 = not torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    output_dir = "outputs_ultra",
    beta = 0.1,
    max_prompt_length = 512,
    max_length = 1024,
)

trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = dpo_config,
    train_dataset = dataset,
    tokenizer = tokenizer,
)

trainer.train()

In [ ]:
FastLanguageModel.for_inference(model) # Включаем режим инференса

prompt = "Кто ты такой?"
inputs = tokenizer(
    [f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
Что ты такое?
assistant
Я искусственный интеллект, созданный компанией Anthropic. Я могу помочь вам с ответами на вопросы, обсуждать различные темы и даже поиграть в игры. Если у вас есть что-то, что вы хотите узнать или обсудить, дайте мне знать!


In [ ]:
!pip install -U trl vllm unsloth

In [ ]:
import re

def format_reward_func(completions, **kwargs):
    """Награда за правильный формат тегов"""
    pattern = r"<think>.*?</think>\s*<answer>.*?</answer>"

    # В новых версиях TRL completions могут приходить в разном формате
    # Проверяем, это список словарей или уже готовые строки
    responses = [
        c[0]["content"] if isinstance(c, list) and isinstance(c[0], dict) else c
        for c in completions
    ]

    return [1.0 if re.search(pattern, r, re.DOTALL) else 0.0 for r in responses]

# ВАЖНО: Заменили queries на prompts (или просто убрали его, оставив **kwargs собирать остатки)
def correctness_reward_func(completions, answer, **kwargs):
    """Награда за правильный математический ответ"""
    responses = [
        c[0]["content"] if isinstance(c, list) and isinstance(c[0], dict) else c
        for c in completions
    ]

    rewards = []
    for response, correct_answer in zip(responses, answer):
        # Ищем то, что внутри <answer>...</answer>
        match = re.search(r"<answer>(.*?)</answer>", response)
        if match and match.group(1).strip() == str(correct_answer).strip():
            rewards.append(2.0)
        else:
            rewards.append(0.0)
    return rewards

In [ ]:
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel

# 1. Загружаем модель
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Для GRPO лучше ранг побольше
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
)

# 2. Загружаем датасет (GSM8K - школьная математика)
dataset = load_dataset("openai/gsm8k", "main", split="train")
def format_dataset(examples):
    # Превращаем обычный текст в формат чата
    return {
        "prompt": [
            [{"role": "user", "content": f"Solve this math problem: {q}"}]
            for q in examples["question"]
        ],
        "answer": examples["answer"] # Оставляем для функции вознаграждения
    }

# Преобразуем датасет
dataset = dataset.map(format_dataset, batched=True)
# 3. Конфиг обучения
training_args = GRPOConfig(
    learning_rate = 5e-6,
    per_device_train_batch_size = 1, # GRPO очень прожорлив до генераций
    gradient_accumulation_steps = 4,
    num_generations = 4, # Сколько ответов сравниваем в группе
    max_prompt_length = 256,
    max_completion_length = 512,
    output_dir = "grpo_outputs",
    fp16 = True,
)

# 4. Запуск
trainer = GRPOTrainer(
    model = model,
    reward_funcs = [format_reward_func, correctness_reward_func],
    args = training_args,
    train_dataset = dataset,
    tokenizer = tokenizer,
)

trainer.train()

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 3 | Total steps = 22,419
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.000000,0.000000,0.000000,377.750000,261.000000,512.000000,0.250000,333.000000,261.000000,455.000000,0.000009,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,369.250000,346.000000,416.000000,0.000000,369.250000,346.000000,416.000000,0.000003,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,278.500000,261.000000,313.000000,0.000000,278.500000,261.000000,313.000000,0.000004,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,348.250000,248.000000,400.000000,0.000000,348.250000,248.000000,400.000000,0.000006,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,201.000000,133.000000,322.000000,0.000000,201.000000,133.000000,322.000000,0.000005,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,467.750000,335.000000,512.000000,0.500000,423.500000,335.000000,512.000000,0.000003,0.000000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,489.500000,450.000000,512.000000,0.500000,467.000000,450.000000,484.000000,0.000008,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,229.750000,79.000000,430.000000,0.000000,229.750000,79.000000,430.000000,0.000010,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,256.500000,202.000000,316.000000,0.000000,256.500000,202.000000,316.000000,0.000006,0.000000,0.000000,0.000000,0.000000
10,0.000000,0.000000,0.000000,335.250000,239.000000,457.000000,0.000000,335.250000,239.000000,457.000000,0.000005,0.000000,0.000000,0.000000,0.000000
